# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [5]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [6]:
get_food_calories('banana')

TypeError: 'FunctionTool' object is not callable

In [8]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories])

In [11]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an pizza?"
    )
    print(result.final_output)

Banana: about 105 calories per medium banana (118 g).

Pizza: please specify type and portion (e.g., one slice of cheese pizza ~285–320 kcal; whole pizza varies widely). If you mean one slice, estimate ~300 kcal.


Enforce tools use:

In [16]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and a slice of dominos chicken pizza?"
    )
    print(result.final_output)

- Banana: about 105 calories (medium, 118 g).
- Domino’s chicken pizza slice: varies by crust, but typically 290–360 calories per slice. Using ~320 kcal per slice gives a total of ~425 calories.

So total is roughly 395–465 calories, depending on crust and slice size. If you specify crust type (hand-tossed vs thin) and slice size, I can give a tighter estimate.
